In [1]:
# 3. Reszta konfiguracji bez zmian
import pyspark
from pyspark.sql import SparkSession


In [2]:
#from pyspark.sql import SparkSession
# 1. Instalacja biblioteki (znak ! wykonuje komendę w systemie)
#!pip install sparksql-magic

#from pyspark.sql import SparkSession

class SparkPostgresConnector:
    def __init__(self, schema_name: str):
        self.spark = SparkSession.builder \
            .appName(f"Connector-{schema_name}") \
            .config("spark.jars.packages", "org.postgresql:postgresql:42.7.1") \
            .getOrCreate()
            
        self.schema = schema_name
        # Upewnij się, że nazwa bazy to dbpostgres, jak widzieliśmy w DBeaverze
        self.jdbc_url = "jdbc:postgresql://postgresdb:5432/dbpostgres"
        
        self.properties = {
            "user": "postgres",
            "password": "password",
            "driver": "org.postgresql.Driver"
        }

    def get_table(self, table_name: str):
        """
        Tworzy połączenie i pobiera dane z konkretnej tabeli w zdefiniowanym schemacie.
        """
        full_table_name = f"{self.schema}.{table_name}"
        
        return self.spark.read.jdbc(
            url=self.jdbc_url,
            table=full_table_name,
            properties=self.properties
        )

    def list_tables(self):
        query = f"(SELECT table_name FROM information_schema.tables WHERE table_schema = '{self.schema}') as tables"
        return self.spark.read.jdbc(url=self.jdbc_url, table=query, properties=self.properties)

    def create_sql_view(self, table_name: str, view_name: str = None):
        """Rejestruje tabelę z Postgresa jako widok SQL w Sparku."""
        if view_name is None:
            view_name = table_name
            
        df = self.get_table(table_name)
        df.createOrReplaceTempView(view_name)
        print(f"✅ Widok '{view_name}' jest gotowy do użycia w SQL.")
# --- Przykład użycia ---
#db = SparkPostgresConnector("projecttables")

# Pobieramy dane z Twojej tabeli 'testtable' widocznej na zrzucie ekranu
#df_test = db.get_table("testtable")

# Teraz możesz wykonywać operacje Sparkowe na tej tabeli
#print("Liczba rekordów w testtable:")
#print(df_test.count())

#df_test.show(5)

In [3]:
# --- Użycie obiektu ---
#db_object = SparkPostgresConnector("projecttables")
#db_object.list_tables().show()

# 1. Inicjalizacja Twojej klasy
db_object = SparkPostgresConnector("projecttables")
spark = db_object.spark

In [4]:
# 1. Rejestrujemy widok (korzystając z Twojej metody w klasie)
db_object.create_sql_view('testtable', 'view_ludzie')

✅ Widok 'view_ludzie' jest gotowy do użycia w SQL.


In [5]:
# Pobieramy dane z Twojej tabeli 'testtable' widocznej na zrzucie ekranu
df_test = db_object.get_table("testtable")

In [6]:
# Teraz możesz wykonywać operacje Sparkowe na tej tabeli
print("Liczba rekordów w testtable:")
print(df_test.count())

df_test.show(5)

Liczba rekordów w testtable:
50
+---+---------+----------+----------+---+------+----+-----------+
| nb|     name|  lastname|     birth|age|weight|high|      pesel|
+---+---------+----------+----------+---+------+----+-----------+
|  1|    Piotr|  Kowalski|1997-05-23| 28| 74.54| 168|97052324430|
|  2|Agnieszka| Kowalczyk|2004-05-26| 21| 76.29| 182|04252635469|
|  3|     Adam| Kowalczyk|1995-08-31| 30| 79.27| 171|95083157030|
|  4|Agnieszka|Wisniewska|2018-11-17|  7| 93.98| 168|18311739480|
|  5|   Tomasz|Wisniewski|1992-09-10| 33| 74.37| 194|92091073076|
+---+---------+----------+----------+---+------+----+-----------+
only showing top 5 rows


In [7]:
# Zapytanie
# 1. Rejestrujesz widok (metoda w klasie wykonuje df.createOrReplaceTempView)
db_object.create_sql_view('testtable', 'testview')

✅ Widok 'testview' jest gotowy do użycia w SQL.


In [8]:
# Zamiast spark.sql(...), użyj sesji wyciągniętej z Twojego obiektu:
db_object.spark.sql("SELECT * FROM testview WHERE age > 25").show()

+---+----------+----------+----------+---+------+----+-----------+
| nb|      name|  lastname|     birth|age|weight|high|      pesel|
+---+----------+----------+----------+---+------+----+-----------+
|  1|     Piotr|  Kowalski|1997-05-23| 28| 74.54| 168|97052324430|
|  3|      Adam| Kowalczyk|1995-08-31| 30| 79.27| 171|95083157030|
|  5|    Tomasz|Wisniewski|1992-09-10| 33| 74.37| 194|92091073076|
|  6|      Anna|     Nowak|1976-05-18| 49| 73.07| 162|76051816368|
|  9|    Tomasz|     Nowak|1988-02-21| 37| 73.79| 162|88022106830|
| 10|     Maria|    Wojcik|1989-04-18| 36| 66.64| 192|89041881223|
| 11|    Tomasz|     Nowak|1995-05-28| 30| 62.34| 172|95052894613|
| 17|      Adam|     Nowak|2000-01-01| 26| 98.82| 166|00210155177|
| 18| Agnieszka|  Kowalska|1999-10-07| 26| 92.43| 187|99100725989|
| 20|      Anna|  Kowalska|1991-05-08| 34| 73.11| 170|91050823582|
| 21|     Piotr|  Kowalski|1978-01-31| 47| 73.09| 185|78013193274|
| 22|      Anna|Wisniewska|1986-08-05| 39| 87.81| 163|86080592